# Day 4 — Multi-Agent Systems and Evaluation

## Daily project: Engineering Design Review Team

This is the classroom master notebook for Day 4. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the environment check and setup cells before beginning.
- Complete sections in order during class; optional provider comparisons are clearly marked.
- If Colab restarts, rerun the current section's import/setup cell before continuing.
- At each checkpoint, explain the observable change before moving forward.
- Use mock or cached mode first. Use the instructor-issued OpenRouter credit only for bounded live observations.

### Day 4 contents

1. [1. A review task we can measure](#day-4-section-1)
2. [2. Single-reviewer baseline](#day-4-section-2)
3. [3. Deterministic tools before model judgment](#day-4-section-3)
4. [4. Parallel specialist reviewers](#day-4-section-4)
5. [5. Supervisor synthesis](#day-4-section-5)
6. [6. Comparative evaluation](#day-4-section-6)
7. [7. Project: Engineering Design Review Team](#day-4-section-7)
8. [Merge Specialist Findings](#day-4-section-8)

---


<a id="day-4-section-1"></a>

## 4.1 — 1. A review task we can measure

“The review sounds good” is not evaluation. This supplied synthetic artifact contains seeded correctness, security, and maintainability defects. First review it without the answer key; later the golden set lets us measure recall and false positives.

The artifact is intentionally unsafe and must never be reused.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Define an evidenced finding and establish a fair hidden-answer review task.

Architecture reference: [Day 4 diagrams D12](../../diagrams/source/day_04.md).

### Expected observation

The artifact prints with line numbers; the golden set contains nine defects but should remain hidden until your first review.

## Concept briefing

## Why multiple agents are not the starting point

Adding agents adds model calls, duplicated context, coordination logic, latency, cost and
new failure modes. It is justified only when a task decomposes into bounded perspectives
whose combined quality exceeds a simpler system by enough to pay for that complexity.

The engineering review project therefore begins with one general reviewer and an
objective artifact containing seeded defects. The single reviewer is allowed to win.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
for number,line in enumerate(SOURCE.splitlines(),1): print(f"{number:>2}: {line}")

## Your independent review

Record each suspected defect as category, line, evidence, severity, and correction. Evidence must point to the artifact; vague style opinions do not count. Only after this attempt, reveal the golden set.

In [ ]:
golden=json.loads(GOLDEN.read_text(encoding="utf-8"))
print("Known defects by category:")
for category in ("correctness","security","maintainability"):
    print(category,[x["id"] for x in golden if x["category"]==category])

## Your turn

Write two findings before revealing the golden file, each with category, line, evidence, severity, and correction.

## Recap

A golden set makes comparison repeatable; it must not leak into the review prompt.

---

### Section 4.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-2"></a>

## 4.2 — 2. Single-reviewer baseline

One reviewer sees the whole artifact and all concerns. Both routes use the same provider contract: the structured mock keeps class reliable, while OpenRouter supplies the live experiment.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Run one general reviewer through a provider contract and preserve its telemetry.

Architecture reference: [Day 4 diagrams D12](../../diagrams/source/day_04.md).

### Expected observation

Mock mode deterministically finds a subset; OpenRouter wording and counts may vary while the finding schema remains fixed.

In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
import os
from review_team import MockStructuredReviewer,OpenRouterReviewer,run_model_review,evaluate
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
single=run_model_review(SOURCE,provider,"general")
for finding in single.findings: print(finding.as_dict())
print(evaluate(single,GOLDEN))

## Inspect the provider boundary

`OpenRouterReviewer` asks for bounded JSON, validates every finding, and records tokens and cost. `MockStructuredReviewer` exercises the same role contract without inference. Neither receives the golden set. Repeat live runs may vary, so preserve each trace.

### Inspect before improving

Which categories were missed? Were claims evidenced? A larger prompt is not automatically a better system; establish the baseline first.

## Your turn

Inspect one missed defect and improve only the prompt or schema—not the answer key.

## Recap

A baseline must exist before adding roles or orchestration.

---

### Section 4.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-3"></a>

## 4.3 — 3. Deterministic tools before model judgment

Parsers, tests, linters, and type checkers provide reproducible evidence. Use them for facts they can establish; reserve model judgment for ambiguity and explanation.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Use AST checks for objective facts and combine them with model judgment.

Architecture reference: [Day 4 diagrams D15](../../diagrams/source/day_04.md).

### Expected observation

The checker finds eval, a mutable default, and a broad exception; synthesis removes overlaps.

## Concept briefing

## Deterministic tools before more model calls

Some findings do not require model judgment. An AST can identify `eval`, mutable default
arguments and broad exceptions reproducibly. Linters, tests, type checkers and security
scanners provide objective evidence for the patterns they support.

Model reviewers are more useful for ambiguous intent, cross-cutting reasoning,
prioritisation and explanation. A strong system combines deterministic evidence with
bounded judgment rather than asking several models to rediscover facts a parser can prove.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
from review_team import deterministic_checks,run_augmented,evaluate
checks=deterministic_checks(SOURCE)
for finding in checks: print(finding.as_dict())
augmented=run_augmented(SOURCE)
print(evaluate(augmented,GOLDEN))
print("Trace:",augmented.trace)

## Boundary

Our small AST checker detects `eval`, mutable defaults, and broad exceptions. It does not prove exploitability or business correctness. In a larger course, pytest, Ruff, Bandit, and mypy could supply additional objective signals—but adding tools without teaching their output would overload this course.

## Your turn

Add one AST check for string-built SQL or explain why a dedicated security tool may be preferable.

## Recap

Use deterministic tools where possible and models where judgment is useful.

---

### Section 4.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-4"></a>

## 4.4 — 4. Parallel specialist reviewers

Multi-agent is useful when work decomposes into independent perspectives. Correctness, security, and maintainability reviewers receive the same immutable artifact and return the same structured contract. Fan-out is capped at three; no reviewer can delegate.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Scope three reviewer roles, compare sequential clarity with parallel fan-out, and preserve structured handoffs.

Architecture reference: [Day 4 diagrams D13](../../diagrams/source/day_04.md).

### Expected observation

Three role traces appear and the supervisor receives only Finding objects.

## Concept briefing

## Specialist decomposition

A specialist role should narrow the task, not merely rename the same prompt. Correctness,
security and maintainability reviewers receive the same immutable artifact but different
evaluation criteria. They return the same `Finding` contract: category, location,
evidence, severity and recommended correction.

Structured handoffs prevent unconstrained agent conversations. The supervisor does not
need every reviewer's full chat history. It needs validated findings and enough provenance
to resolve duplicates and conflicts.

## Sequential before parallel

Run specialists sequentially first because the execution order and failures are easy to
inspect. If the branches are independent, they can then fan out in parallel and fan in at
the supervisor. Parallelism may reduce wall-clock time but does not reduce total model
calls or tokens. It may also trigger provider rate limits.

The fan-in step must be bounded. It validates fields, deduplicates, ranks, caps output and
terminates. A supervisor that can indefinitely request revisions has created another
autonomous loop rather than a controlled aggregation step.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
from review_team import specialist_review
categories=["correctness","security","maintainability"]
groups={category:specialist_review(SOURCE,category) for category in categories}
for category,findings in groups.items():
    print("\n",category)
    for finding in findings: print(finding.as_dict())

## Why parallel?

These branches do not depend on one another, so they may run concurrently and later fan in. Parallelism can reduce wall time with hosted APIs, but raises calls, tokens, rate-limit pressure, and debugging complexity. Local code may be too fast for timing differences to matter.

In [ ]:
import os
from review_team import MockStructuredReviewer,OpenRouterReviewer,run_model_multi
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
multi=run_model_multi(SOURCE,provider)
print("Calls:",multi.model_calls,"tokens (input estimate):",multi.estimated_tokens)
print("Trace:",multi.trace)

## Optional direct LangGraph

Represent state as artifact plus lists of structured findings. Add three reviewer nodes from `START`, connect all to one supervisor, and compile. Use a reducer for concurrently returned lists. LangGraph coordinates state; it does not make reviewer judgment correct.

## Your turn

Run the same roles sequentially first; then compare calls, results, and wall time with fan-out.

## Recap

Multi-agent means bounded decomposition, not unrestricted agent conversation.

---

### Section 4.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-5"></a>

## 4.5 — 5. Supervisor synthesis

Fan-out creates duplicates and inconsistent severity. The supervisor’s narrow job is to validate fields, deduplicate by stable identity, rank, cap output, and terminate. It does not start another open-ended conversation.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Validate, deduplicate, rank, cap, and terminate specialist findings.

Architecture reference: [Day 4 diagrams D14](../../diagrams/source/day_04.md).

### Expected observation

Duplicate stable IDs collapse and critical findings appear before medium ones.

## Concept briefing

## Deduplication is harder than matching IDs

Stable seeded IDs make the classroom evaluator simple. Real reviewers may describe the
same issue with different titles or identify one root cause at different lines. Similarity
can help group candidates, but a human may still need to resolve ambiguous merges. The
course's deterministic deduplication demonstrates orchestration and should not be mistaken
for a complete production finding-resolution system.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
from review_team import deterministic_checks,specialist_review,synthesize
groups=[deterministic_checks(SOURCE)] + [specialist_review(SOURCE,c) for c in ("correctness","security","maintainability")]
print("Before fan-in:",sum(map(len,groups)))
final=synthesize(groups,max_findings=20)
print("After deduplication:",len(final))
for finding in final: print(finding.severity,finding.id,finding.title)

## Handoffs are contracts

Only `Finding` objects cross the boundary—not personas, hidden chain-of-thought, or full conversations. Stable IDs make deduplication easy in this seeded lab. Real systems need a similarity rule plus human review because two differently worded findings may describe one root cause.

## Your turn

Create two differently worded findings at the same line and document why stable-ID deduplication is insufficient.

## Recap

A supervisor has a narrow aggregation contract and a stopping condition.

---

### Section 4.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-6"></a>

## 4.6 — 6. Comparative evaluation

Run all systems on the same artifact and answer with measurements: Did specialization improve defect recall enough to justify extra calls, tokens, latency, cost, and operational complexity? The single reviewer is allowed to win.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Calculate recall, false positives, calls, tokens, latency, and cost on the same artifact.

Architecture reference: [Day 4 diagrams D15](../../diagrams/source/day_04.md).

### Expected observation

Offline orchestration yields 5/9, 6/9, and 9/9; live model results may vary and must be preserved.

## Concept briefing

## Evaluating nondeterministic systems

Do not assert exact model wording. Test invariants and outcomes:

- Is every finding structurally valid?
- Does evidence refer to the supplied artifact?
- How many known defects were found?
- How many unsupported findings were reported?
- How many duplicates survived synthesis?
- How many calls and tokens were used?
- Did the system terminate within its bounds?

One run is an anecdote. Repeat model experiments with the same model, prompt version,
temperature and artifact. Report variance rather than selecting the best result.

## Capability can change the architecture conclusion

A weaker instruction-following model may benefit disproportionately from narrow prompts.
A stronger model may handle the general review well enough that specialist calls add
little value. Therefore "multi-agent is better" may actually mean "decomposition
compensated for this model under this task and prompt."

An instructor may repeat the same golden-set experiment on a currently strong reference
model. The lesson is not brand ranking. It is that model capability, cost and reliability
are architecture inputs.

## Cost and latency arithmetic

Approximate run cost as:

```text
sum of input tokens across calls
+ sum of output and reasoning tokens
+ retries
```

If the same 1,000-token artifact is sent to three specialists, the input is paid three
times unless caching or a provider feature changes the calculation. Parallel execution
may reduce elapsed time while preserving or increasing total cost.

A fair comparison records recall, false positives, calls, tokens, latency, estimated cost
and debugging complexity. The chosen system should be the smallest one that meets the
quality requirement.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
import os
from review_team import *
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
runs=[run_model_review(SOURCE,provider,"general"),run_augmented(SOURCE),run_model_multi(SOURCE,provider)]
rows=[evaluate(run,GOLDEN,price_per_million_tokens=0.0) for run in runs]
headers=["system","found","recall","false_positives","duplicates","model_calls","estimated_tokens","elapsed_ms","estimated_cost_usd"]
print(" | ".join(headers))
for row in rows: print(" | ".join(str(row[h]) for h in headers))

## Interpret carefully

Our offline specialists encode known category patterns, so their 100% result validates orchestration—not general model intelligence. A live A/B should hide the golden set, repeat trials, pin model/configuration, and report variance. Token counts here approximate repeated source input; provider usage is preferable for live runs.

In [ ]:
for row in rows: print(row["system"],"missed:",row["missed"])
best=max(rows,key=lambda r:(r["recall"],-r["model_calls"]))
print("Best under recall-then-fewer-calls rule:",best["system"])

## Required live observation

Run one single-reviewer and one bounded specialist comparison with the issued model. Preserve raw structured results; use the captured comparison if the service is unavailable.


## Your turn

Hand-calculate recall for one run, then repeat a model A/B twice and report variance.

## Recap

Choose the smallest system that meets measured quality requirements.

---

### Section 4.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-7"></a>

## 4.7 — 7. Project: Engineering Design Review Team

Demonstrate three bounded systems, inspect their traces, and defend a deployment choice. The goal is not “more agents”; it is the smallest system whose measured quality meets the requirement.


## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Integrate provider-backed roles, deterministic checks, supervisor synthesis, traces, and a decision memo.

Architecture reference: [Day 4 diagrams D12–D15](../../diagrams/source/day_04.md).

### Expected observation

Both single and specialist systems terminate with structured findings and comparable telemetry.

## Concept briefing

## What to carry into Day 5

Days 1-4 repeatedly configure providers, validate tools, enforce limits and record events.
Day 5 extracts these repeated responsibilities into reusable infrastructure while keeping
application-specific instructions, tools and policy in agent configurations.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
import os
from review_team import *
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
systems=[run_model_review(SOURCE,provider,"general"),run_augmented(SOURCE),run_model_multi(SOURCE,provider)]
reports=[]
for run in systems:
    report=evaluate(run,GOLDEN); reports.append(report)
    print("\nSYSTEM",run.system,report)
    for event in run.trace: print(" ",event)

In [ ]:
assert reports[0]["model_calls"]==1 and reports[2]["model_calls"]==3
assert all(0.0<=r["recall"]<=1.0 for r in reports)
print("Structural comparison checks passed; quality is an observed result, not an assertion.")

## Decision memo

Submit one paragraph naming your chosen system and evidence. Include recall, false positives, calls/tokens, latency caveats, cost assumption, and debugging burden. Then describe one condition that would reverse your choice.

### Repeat the experiment

When using OpenRouter, repeat both provider-backed systems with the same model and configuration. Keep prompts scoped, output bounded, and raw traces saved locally or optionally in LangSmith using only the supplied artifact. Report variance rather than selecting the best single run.

## Your turn

Submit one choice and one condition that would reverse it; include raw trace evidence.

## Recap

More agents are justified only by measured benefit.

---

### Section 4.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-8"></a>

## 4.8 — Merge Specialist Findings

This is an individual implementation lab. It uses no API key.


## Why this mechanism matters

Specialists may overlap, disagree, or fail. Deterministic aggregation makes the supervisor boundary inspectable and avoids spending another model call on rules ordinary code can enforce.

## Contract

Ignore failed specialist results, keep the first copy of each finding ID, and sort successful findings by descending severity then ascending ID.

Before coding, write one sentence predicting the easiest failure to make.

In [ ]:
def merge_findings(results):
    # TODO: tolerate status == "error"
    # TODO: deduplicate by stable ID
    # TODO: sort by (-severity, id)
    raise NotImplementedError("Complete supervisor merge")

## Behavioural check

Run this only after completing the starter cell. A passing check proves the listed contract examples, not every possible input.

In [ ]:
results = [
    {"status": "ok", "findings": [{"id": "F2", "severity": 2}, {"id": "F1", "severity": 3}]},
    {"status": "error", "error": "timeout"},
    {"status": "ok", "findings": [{"id": "F1", "severity": 3}, {"id": "F3", "severity": 1}]},
]
merged = merge_findings(results)
assert [item["id"] for item in merged] == ["F1", "F2", "F3"]
print(merged); print("PASS")

## Explain and extend

Why can ID-based deduplication still miss semantic duplicates? Add a second deterministic key using category and line number, then describe its possible false merges.

---

### Section 4.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 4 completion checklist

- [ ] I can explain how every section contributes to the **Engineering Design Review Team**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I completed the pivotal exercise without copying the reference implementation.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
